In [8]:
%run ./setup_data.ipynb

Old Spark session successfully stopped.
Starting new Spark session: 'working_with_different_data_type'...
Creating payment table
Configured Warehouse Path: file:/home/suanki/Desktop/projects/suanki-pyspark-and-optimization/aggregation/spark-warehouse
Target Absolute Path:      /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/aggregation/spark-warehouse/sujeet_db.db/payment
Status: Path found! Proceeding with deletion for payment...
Success: Directory deleted successfully.
Creating sales table for variance analysis
Configured Warehouse Path: file:/home/suanki/Desktop/projects/suanki-pyspark-and-optimization/aggregation/spark-warehouse
Target Absolute Path:      /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/aggregation/spark-warehouse/sujeet_db.db/sales_metrics
Status: Path found! Proceeding with deletion for sales_metrics...
Success: Directory deleted successfully.
Sales metrics table created successfully.
Creating skewed data table...
Configured Warehouse P

## Aggregations

The simplest grouping is to just summarize a complete DataFrame by performing an
aggregation in a select statement.

- A “group by” allows you to specify one or more keys as well as one or more aggregation functions to transform the value columns.
- A “window” gives you the ability to specify one or more keys as well as one or more aggregation functions to transform the value columns. However, the rows input to the function are somehow related to the current row.
- A “grouping set,” which you can use to aggregate at multiple different levels. Grouping sets are available as a primitive in SQL and via rollups and cubes in DataFrames.
- A “rollup” makes it possible for you to specify one or more keys as well as one or more aggregation functions to transform the value columns, which will be summarized hierarchically.
- A “cube” allows you to specify one or more keys as well as one or more aggregation functions to transform the value columns, which will be summarized across all combinations of columns.


Each grouping returns a RelationalGroupedDataset on which we specify our aggregations.



In [9]:
spark

In [10]:
spark.sql(""" show databases""").show()
spark.sql(""" show tables""").show()

department_df = obj.prepareData(spark)
payement_df = spark.sql("""select * from sujeet_db.payment""")
sales_df = spark.sql("""select * from sujeet_db.sales_metrics""")
skew_matrics_df = spark.sql("""select * from sujeet_db.skew_metrics""")
weather_sales_df = spark.sql("""select * from sujeet_db.weather_sales""")

+---------+
|namespace|
+---------+
|  default|
|sujeet_db|
+---------+

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|sujeet_db|      payment|      false|
|sujeet_db|sales_metrics|      false|
|sujeet_db| skew_metrics|      false|
|sujeet_db|weather_sales|      false|
+---------+-------------+-----------+

creating sample dataframe


In [ ]:
department_df.count()

In [ ]:
department_df.count() == 12

In [ ]:
### Aggregation function

from pyspark.sql.functions import count, min, max, avg, first, last, sum, sum_distinct

## count(*) count null as one record
department_df.select(
    count("department"), 
    count("*"),
    count("salary"),  
    min("salary"), 
    max("salary"), 
    avg("salary"),
    first("department"),
    last("department"),
    sum("salary"),
    sum_distinct("salary"),
    ).show()

In [ ]:
from pyspark.sql.functions import count_distinct
department_df.select("department").distinct().show()

## countDistinct does not count null 

department_df.select(count_distinct("department"), count("department")).show()

### **The Definitions**
* **The Variance ($\sigma^2$):** The average of the squared differences from the Mean.
* **The Standard Deviation ($\sigma$):** The square root of the variance. This returns the metric to the original unit of measure.

---

### **Example Scenario**
Let's analyze a simple dataset representing the daily sales (in thousands) of a retail store over a 5-day period:
$$\text{Dataset } (X) = [2, 4, 4, 4, 6]$$

---

### **Step 1: Calculate the Mean ($\mu$)**
The mean is the basic average of all data points.

$$\mu = \frac{\sum X}{N} = \frac{2 + 4 + 4 + 4 + 6}{5} = \frac{20}{5} = 4$$

---

### **Step 2 & 3: Find the Squared Differences from the Mean**
Subtract the mean ($4$) from every data point, and square the resulting value to eliminate negative signs.

| Original Value ($X$) | Difference from Mean ($X - \mu$) | Squared Difference $(X - \mu)^2$ |
| :--- | :--- | :--- |
| 2 | $2 - 4 = -2$ | $(-2)^2 = 4$ |
| 4 | $4 - 4 = 0$ | $(0)^2 = 0$ |
| 4 | $4 - 4 = 0$ | $(0)^2 = 0$ |
| 4 | $4 - 4 = 0$ | $(0)^2 = 0$ |
| 6 | $6 - 4 = 2$ | $(2)^2 = 4$ |

---

### **Step 4: Calculate the Variance ($\sigma^2$)**
Find the average of the values in the third column by summing them up and dividing by the total count ($N = 5$).

$$\text{Variance } (\sigma^2) = \frac{4 + 0 + 0 + 0 + 4}{5} = \frac{8}{5} = 1.6$$

---

### **Step 5: Calculate the Standard Deviation ($\sigma$)**
Take the square root of the variance value calculated above.

$$\text{Standard Deviation } (\sigma) = \sqrt{1.6} \approx 1.265$$

---

### **📊 Analytical Interpretation**
* **Mean ($4$):** On an average day, the store brings in $4,000 in sales.
* **Standard Deviation ($1.265$):** Most typical daily sales figures deviate or scatter away from that $4,000

center mark by roughly 1,265.


In [ ]:
sales_df.show()

In [ ]:
from pyspark.sql.functions import avg, var_pop, stddev_pop

(
    sales_df
    .select(
        avg("daily_sales").alias("avg_sales"),
        var_pop("daily_sales").alias("variance"),
        stddev_pop("daily_sales").alias("standard_deviation")
    )
).show(10, False)


1. Skewness (Assymmetry)

Skewness measures the symmetry of a distribution. A perfectly symmetrical distribution (like a standard bell curve) has a skewness of 0.
• Positive Skew (Right-skewed): The tail extends longer on the right side. Most values are bunched on the left, but a few unusually high values drag the average up. (Example: House prices or incomes).
• Negative Skew (Left-skewed): The tail extends longer on the left side. Most values are bunched on the right, but a few unusually low values drag the average down. (Example: Age of retirement).

2. Kurtosis (Tails and Peakedness)

Kurtosis measures the heaviness of the tails (how often extreme outliers occur) compared to a normal distribution. A standard normal distribution has a Pearson Kurtosis of 3 (or an Excess Kurtosis of 0).
• High Kurtosis (> 3): Heavy tails. This means the data has sharp spikes and an increased probability of extreme outliers ("fat tails").
• Low Kurtosis (< 3): Light tails. The distribution is flatter with fewer extreme outliers.


# Comprehensive Math Reference: Skewness & Kurtosis

This notebook provides the mathematical definitions, formulas, and step-by-step calculations for **Skewness** and **Kurtosis** using a sample dataset.

---

### **The Math Formulas**
To calculate these metrics, we normalize the differences from the mean using the standard deviation ($\sigma$). This is known as calculating **standardized moments**.

* **Skewness ($G_1$):** The average of the cubed standardized differences.
  $$G_1 = \frac{1}{N} \sum \left( \frac{X - \mu}{\sigma} \right)^3$$

* **Kurtosis ($G_2$):** The average of the fourth-powered standardized differences.
  $$G_2 = \frac{1}{N} \sum \left( \frac{X - \mu}{\sigma} \right)^4$$

---

### **Example Scenario & Setup**
Let's use a highly skewed 5-element dataset representing a sudden surge in product sales:
$$\text{Dataset } (X) = [2, 2, 2, 2, 12]$$

From this dataset, we can pre-calculate the fundamental baselines:
* **Total Items ($N$):** $5$
* **Mean ($\mu$):** $\frac{2 + 2 + 2 + 2 + 12}{5} = \frac{20}{5} = 4$
* **Population Variance ($\sigma^2$):** $\frac{4 \times (2-4)^2 + (12-4)^2}{5} = \frac{4(4) + 64}{5} = \frac{80}{5} = 16$
* **Population Standard Deviation ($\sigma$):** $\sqrt{16} = 4$

---

### **Step-by-Step Calculation Table**

We convert each raw value into its standardized score ($Z = \frac{X - \mu}{\sigma}$), then calculate the $3^{\text{rd}}$ power (for Skewness) and $4^{\text{th}}$ power (for Kurtosis).

| $X$ | Diff ($X - \mu$) | Std Score $\left(\frac{X - \mu}{\sigma}\right)$ | Cubed Score $\left(\frac{X - \mu}{\sigma}\right)^3$ | $4^{\text{th}}$ Power $\left(\frac{X - \mu}{\sigma}\right)^4$ |
| :--- | :--- | :--- | :--- | :--- |
| 2 | $2 - 4 = -2$ | $\frac{-2}{4} = -0.5$ | $(-0.5)^3 = -0.125$ | $(-0.5)^4 = 0.0625$ |
| 2 | $2 - 4 = -2$ | $\frac{-2}{4} = -0.5$ | $(-0.5)^3 = -0.125$ | $(-0.5)^4 = 0.0625$ |
| 2 | $2 - 4 = -2$ | $\frac{-2}{4} = -0.5$ | $(-0.5)^3 = -0.125$ | $(-0.5)^4 = 0.0625$ |
| 2 | $2 - 4 = -2$ | $\frac{-2}{4} = -0.5$ | $(-0.5)^3 = -0.125$ | $(-0.5)^4 = 0.0625$ |
| 12 | $12 - 4 = 8$ | $\frac{8}{4} = 2.0$ | $(2.0)^3 = 8.000$ | $(2.0)^4 = 16.0000$ |
| **Sum**| | | **$7.5$** | **$16.25$** |

---

### **Final Results**

#### **1. Skewness Calculation**
Take the average of the Cubed Scores column:
$$\text{Skewness} = \frac{7.5}{5} = 1.5$$
* **Interpretation:** The value is positive ($1.5 > 0$), confirming a **strong positive (right) skew** driven by the extreme outlier ($12$).

#### **2. Kurtosis Calculation**
Take the average of the $4^{\text{th}}$ Power column:
$$\text{Kurtosis} = \frac{16.25}{5} = 3.25$$
* **Interpretation:** The value is greater than a standard normal distribution ($3.25 > 3$). This indicates a **leptokurtic** distribution with a sharp peak and a heavy tail caused by the outlier.


In [ ]:
skew_matrics_df.show()

In [ ]:
from pyspark.sql.functions import skewness, kurtosis

skew_matrics_df.select(skewness("item_value"), kurtosis("item_value")).show()

## Covariance

Covariance indicates the **direction** of the relationship between two variables.

- **Positive covariance:** As \(X\) increases, \(Y\) tends to increase (e.g., study hours and exam scores).
- **Negative covariance:** As \(X\) increases, \(Y\) tends to decrease (e.g., screen time and sleep hours).
- **Limitation:** Covariance depends on the variables’ units. For example, changing dollars to cents changes the covariance, making the strength of the relationship difficult to compare.

## Correlation (Pearson’s \(r\))

Correlation standardizes covariance to describe both the **direction** and **strength** of a linear relationship.

- It divides covariance by the standard deviations of both variables.
- Its value ranges from \(-1\) to \(+1\):
	- \(+1\): Perfect positive linear relationship
	- \(-1\): Perfect negative linear relationship
	- \(0\): No linear relationship

# Comprehensive Math Reference: Covariance & Correlation

This notebook outlines the step-by-step mathematical logic for calculating population Covariance and Correlation between two variables.

---

### **The Math Formulas**
* **Covariance ($\text{Cov}(X,Y)$):** The average of the product of the differences from their respective means.
  $$\text{Cov}(X,Y) = \frac{1}{N} \sum (X - \mu_X)(Y - \mu_Y)$$

* **Correlation ($r$):** The covariance divided by the product of their standard deviations.
  $$r = \frac{\text{Cov}(X,Y)}{\sigma_X \times \sigma_Y}$$

---

### **Example Scenario**
Let's measure the relationship between **Daily Temperature ($X$)** and **Ice Cream Sales ($Y$)** over 5 days:
* $\text{Dataset } X = [2, 4, 4, 4, 6]$ *(Mean $\mu_X = 4$, Std Dev $\sigma_X \approx 1.265$, Var $\sigma^2_X = 1.6$)*
* $\text{Dataset } Y = [10, 20, 25, 25, 40]$

---

### **Step 1: Baseline Metrics for $Y$**
* **Mean ($\mu_Y$):** $\frac{10 + 20 + 25 + 25 + 40}{5} = \frac{120}{5} = 24$
* **Variance ($\sigma^2_Y$):** $\frac{(10-24)^2 + (20-24)^2 + 2(25-24)^2 + (40-24)^2}{5} = \frac{196 + 16 + 2 + 256}{5} = \frac{470}{5} = 94$
* **Standard Deviation ($\sigma_Y$):** $\sqrt{94} \approx 9.695$

---

### **Step 2: Step-by-Step Calculation Table**

| Day | $X$ | $Y$ | $X - \mu_X$ | $Y - \mu_Y$ | Product: $(X - \mu_X)(Y - \mu_Y)$ |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | 2 | 10 | $2 - 4 = -2$ | $10 - 24 = -14$ | $(-2) \times (-14) = 28$ |
| 2 | 4 | 20 | $4 - 4 = 0$ | $20 - 24 = -4$ | $0 \times (-4) = 0$ |
| 3 | 4 | 25 | $4 - 4 = 0$ | $25 - 24 = 1$ | $0 \times 1 = 0$ |
| 4 | 4 | 25 | $4 - 4 = 0$ | $25 - 24 = 1$ | $0 \times 1 = 0$ |
| 5 | 6 | 40 | $6 - 4 = 2$ | $40 - 24 = 16$ | $2 \times 16 = 32$ |
| **Sum** | | | | | **$60$** |

---

### **Final Results**

#### **1. Population Covariance Calculation**
Take the average of the Product column:
$$\text{Cov}(X,Y) = \frac{60}{5} = 12$$
* **Interpretation:** The value is positive ($12 > 0$), meaning as temperature increases, ice cream sales move up in the same direction.

#### **2. Population Correlation Calculation ($r$)**
Divide the covariance by the product of both standard deviations ($\sigma_X \approx 1.265$ and $\sigma_Y \approx 9.695$):
$$r = \frac{12}{1.265 \times 9.695} = \frac{12}{12.264} \approx 0.978$$
* **Interpretation:** $0.978$ indicates an **extremely strong positive linear relationship**. The temperature perfectly tracks changes in sales.


In [4]:
weather_sales_df.show()

+------+-----------+---------------+
|day_id|temperature|ice_cream_sales|
+------+-----------+---------------+
|     4|       4.00|          25.00|
|     5|       6.00|          40.00|
|     2|       4.00|          20.00|
|     1|       2.00|          10.00|
|     3|       4.00|          25.00|
+------+-----------+---------------+



In [6]:
from pyspark.sql.functions import corr, covar_pop, covar_samp
(
    weather_sales_df
    .select(
        corr("temperature", "ice_cream_sales").alias("correlation"),
        covar_pop("temperature", "ice_cream_sales").alias("population_covariance"),
        covar_samp("temperature", "ice_cream_sales").alias("sample_covariance")
    )
).show()




+------------------+---------------------+------------------+
|       correlation|population_covariance| sample_covariance|
+------------------+---------------------+------------------+
|0.9784921095801634|   12.000000000000002|15.000000000000002|
+------------------+---------------------+------------------+



In [11]:
spark.stop()